# Ранжирование фрагментов AMML 2026

Здесь только итоговый пайплайн, где мы дообучаем Qwen, Yuanbao и Giga, считаем предсказания и объединяем их по рангу в ансамбль. Исходный запуск выполнялся на **Kaggle (Linux)** на доступных там двух T4, но я постаралась сделать так, чтобы код на всякий случай не ломался и на 1 GPUшке


## Данные и окружение

In [ ]:
import os
import sys
import subprocess
import zipfile
import shutil
from pathlib import Path

import pandas as pd

project_dir = Path.cwd().resolve()
default_data = (Path('/kaggle/input') if Path('/kaggle/input').exists()
                else project_dir / 'data' if (project_dir / 'data').exists()
                else project_dir)
data_dir = Path(os.environ.get('AMML_DATA_ROOT', default_data)).resolve()
work_dir = Path(os.environ.get(
    'AMML_WORK_DIR',
    '/kaggle/working' if Path('/kaggle/working').exists() else project_dir / 'outputs'
)).resolve()
work_dir.mkdir(parents=True, exist_ok=True)

pred_dir = work_dir / 'predictions'
pred_dir.mkdir(exist_ok=True)

run_env = dict(
    os.environ,
    TOKENIZERS_PARALLELISM='false',
    HF_HOME=os.environ.get('HF_HOME', str(work_dir / 'hf_cache')),
    MODELSCOPE_CACHE=os.environ.get('MODELSCOPE_CACHE', str(work_dir / 'modelscope_cache')),
)


def run(script):
    subprocess.run([sys.executable, script], cwd=work_dir, env=run_env, check=True)

In [ ]:
from importlib.metadata import PackageNotFoundError, version

try:
    version('torchao')
except PackageNotFoundError:
    pass
else:
    raise RuntimeError('Запустите ноутбук в отдельном окружении без torchao (см. README).')

In [ ]:
import torch

if not torch.cuda.is_available():
    raise RuntimeError('Для обучения нужна доступная CUDA GPU')

In [ ]:
filenames = ('train.csv', 'test.csv')
source_dir = work_dir / 'data'

pairs = [p.parent for p in sorted(data_dir.rglob('train.csv'))
         if (p.parent / 'test.csv').is_file() and p.parent != source_dir]
archives = sorted(data_dir.rglob('amml-ranking-competition-2026.zip'))

source_dir.mkdir(exist_ok=True)
if all((data_dir / name).is_file() for name in filenames):
    input_dir = data_dir
elif len(pairs) == 1 and not archives:
    input_dir = pairs[0]
elif len(archives) == 1:
    input_dir = None
else:
    raise FileNotFoundError(
        'Укажите AMML_DATA_ROOT с одной парой train.csv/test.csv или одним архивом AMML 2026.'
    )

if input_dir is None:
    with zipfile.ZipFile(archives[0]) as archive:
        members = {
            Path(name).name: name for name in archive.namelist()
            if Path(name).name in filenames and '__MACOSX' not in Path(name).parts
        }
        if members.keys() != set(filenames):
            raise FileNotFoundError('В архиве нужны train.csv и test.csv.')
        for name in filenames:
            with archive.open(members[name]) as src, (source_dir / name).open('wb') as dst:
                shutil.copyfileobj(src, dst)
elif input_dir != source_dir:
    for name in filenames:
        shutil.copyfile(input_dir / name, source_dir / name)

test_raw = pd.read_csv(source_dir / 'test.csv', dtype={'id': str})

In [ ]:
%%writefile "{work_dir}/amml_common.py"
import html
import re

import pandas as pd

# Те же версии базовых моделей, что в исходных запусках Kaggle
QWEN = ('Qwen/Qwen3-Reranker-8B', '77d193c791ed757ca307ee72715aa132723da912')
YUANBAO = ('Yuanbao-Search/Yuanbao-Reranker-8B', '3868680274e1914e7b544e32fd36727b1c679fce')
GIGA = ('ai-sage/Giga-Embeddings-instruct-3B-0826', 'b71168088212f0a13688514e4fc288a86106c4e4')


def clean(value):
    if pd.isna(value):
        return ''
    s = html.unescape(str(value)).replace('\u0301', '').replace('\u200b', '')
    s = re.sub(r'<[^>]{1,120}>', ' ', s)
    return re.sub(r'\s+', ' ', s).strip()


def prepare(df):
    df = df.copy()
    for source in ('query', 'title', 'text'):
        df[source] = df[source].map(clean)
    df['title'] = df.title.str.replace(r'\s*[-—–]\s*Википедия\s*$', '', regex=True, case=False)
    df['doc'] = 'Название статьи: ' + df.title + '\nФрагмент: ' + df.text
    return df


def load_reranker(model_name, revision):
    import torch
    from transformers import AutoModelForCausalLM, BitsAndBytesConfig

    quantization = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type='nf4',
        bnb_4bit_use_double_quant=True,
        bnb_4bit_compute_dtype=torch.float16,
    )
    two_gpus = torch.cuda.device_count() >= 2
    device_map = dict.fromkeys(
        ('model.embed_tokens', 'model.rotary_emb', 'model.norm', 'lm_head'), 0
    )
    for layer in range(36):
        device_map[f'model.layers.{layer}'] = int(two_gpus and layer >= 18)
    model = AutoModelForCausalLM.from_pretrained(
        model_name, revision=revision, device_map=device_map,
        quantization_config=quantization, dtype=torch.float16,
        attn_implementation='sdpa', trust_remote_code=False,
    )
    model.config.use_cache = False
    return model

## Qwen3-Reranker-8B


In [ ]:
%%writefile "{work_dir}/qwen_train.py"
import random
import re
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from peft import LoraConfig, TaskType, get_peft_model, prepare_model_for_kbit_training
from sklearn.model_selection import GroupShuffleSplit
from transformers import AutoTokenizer

from amml_common import QWEN, load_reranker, prepare

MODEL_NAME, REVISION = QWEN
SEED = 42
MAX_LENGTH = 1024
OUT = Path('qwen')
OUT.mkdir(exist_ok=True)

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

In [ ]:
%%writefile -a "{work_dir}/qwen_train.py"
# Разделение и отбор примеров
train = prepare(pd.read_csv('data/train.csv'))

split = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=SEED)
fit_idx, val_idx = next(split.split(train, groups=train.query_id))
fit, valid = train.iloc[fit_idx].copy(), train.iloc[val_idx].copy()

stop = {'как', 'что', 'кто', 'где', 'когда', 'какой', 'какие', 'в', 'на', 'с', 'из', 'по', 'и', 'или'}
def similar_words(query, doc):
    q = {w for w in re.findall(r'\w+', query.casefold()) if len(w) > 2 and w not in stop}
    d = set(re.findall(r'\w+', doc.casefold()))
    return len(q & d) / max(len(q), 1)

rng = np.random.default_rng(SEED)
chosen = []
for _, group in fit.groupby('query_id', sort=True):
    pos = group[group.relevance == 1]
    neg = group[group.relevance == 0].copy()
    chosen.extend(pos.index.tolist())
    if neg.empty:
        continue
    neg['overlap'] = [similar_words(q, d) for q, d in zip(neg['query'], neg['doc'])]
    neg = neg.sort_values(['overlap', 'id'], ascending=[False, True])
    pos_articles = set(pos.article_id)
    same_article = neg[neg.article_id.isin(pos_articles)]
    if len(same_article):
        idx = same_article.index[0]
        chosen.append(idx)
        neg = neg.drop(idx)
    if len(neg):
        idx = neg.index[0]
        chosen.append(idx)
        neg = neg.drop(idx)
    if len(neg):
        chosen.append(rng.choice(neg.index))

fit = fit.loc[chosen].sample(frac=1, random_state=SEED).reset_index(drop=True)
print(f'Train: {len(fit):,} пар, validation: {len(valid):,} пар', flush=True)

In [ ]:
%%writefile -a "{work_dir}/qwen_train.py"
# Токенизация
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, revision=REVISION)
no_id = tokenizer.convert_tokens_to_ids('no')
yes_id = tokenizer.convert_tokens_to_ids('yes')

prefix = ('<|im_start|>system\nJudge whether the Document meets the requirements '
          'based on the Query and the Instruct provided. Note that the answer '
          'can only be "yes" or "no".<|im_end|>\n<|im_start|>user\n')
suffix = '<|im_end|>\n<|im_start|>assistant\n<think>\n\n</think>\n\n'
prefix_ids = tokenizer.encode(prefix, add_special_tokens=False)
suffix_ids = tokenizer.encode(suffix, add_special_tokens=False)


def encode(query, doc):
    content = (f'<Instruct>: Given a web search query, retrieve relevant passages '
               f'that answer the query\n<Query>: {query}\n<Document>: {doc}')
    room = MAX_LENGTH - len(prefix_ids) - len(suffix_ids)
    middle = tokenizer.encode(content, add_special_tokens=False, truncation=True, max_length=room)
    return prefix_ids + middle + suffix_ids

train_tokens = [encode(q, d) for q, d in zip(fit['query'], fit['doc'])]
labels = fit.relevance.to_numpy(dtype=np.int64)

In [ ]:
%%writefile -a "{work_dir}/qwen_train.py"
# Загрузка 4-битной модели и LoRA
model = load_reranker(MODEL_NAME, REVISION)
model = prepare_model_for_kbit_training(
    model, use_gradient_checkpointing=True,
    gradient_checkpointing_kwargs={'use_reentrant': False}
)
model = get_peft_model(model, LoraConfig(
    r=16, lora_alpha=32, lora_dropout=0.05, bias='none',
    target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj', 'gate_proj', 'up_proj', 'down_proj'],
    task_type=TaskType.CAUSAL_LM,
))
input_device = model.get_input_embeddings().weight.device

In [ ]:
%%writefile -a "{work_dir}/qwen_train.py"
# Валидация
val_ids = np.random.default_rng(SEED).choice(
    valid['query_id'].unique(), size=min(96, valid['query_id'].nunique()), replace=False
)
val_small = valid[valid['query_id'].isin(val_ids)].copy()
val_tokens = [encode(q, d) for q, d in zip(val_small['query'], val_small['doc'])]


def ndcg_at_10(df, scores):
    data = df[['query_id', 'id', 'relevance']].copy()
    data['score'] = scores
    data['id'] = data['id'].astype(str)
    values = []
    for _, group in data.groupby('query_id', sort=False):
        ranked = group.sort_values(['score', 'id'], ascending=[False, True])
        gains = ranked['relevance'].to_numpy()[:10]
        ideal = np.sort(group['relevance'].to_numpy())[::-1][:10]
        discounts = np.log2(np.arange(2, len(gains) + 2))
        dcg = np.sum((2 ** gains - 1) / discounts)
        ideal_discounts = np.log2(np.arange(2, len(ideal) + 2))
        idcg = np.sum((2 ** ideal - 1) / ideal_discounts)
        values.append(dcg / idcg if idcg else 0.0)
    return float(np.mean(values))


def validate():
    model.eval()
    scores = []
    with torch.inference_mode():
        for tokens in val_tokens:
            ids = torch.tensor([tokens], dtype=torch.long, device=input_device)
            with torch.autocast('cuda', dtype=torch.float16):
                logits = model(input_ids=ids, logits_to_keep=1, use_cache=False).logits[0, -1]
            scores.append((logits[yes_id] - logits[no_id]).float().item())
    model.train()
    return ndcg_at_10(val_small, scores)

In [ ]:
%%writefile -a "{work_dir}/qwen_train.py"
# Обучение и сохранение адаптера
ACCUMULATION = 8
PATIENCE = 2
MIN_DELTA = 0.001

steps_per_epoch = (len(train_tokens) + ACCUMULATION - 1) // ACCUMULATION
check_steps = sorted({max(1, round(steps_per_epoch * x)) for x in (0.25, 0.50, 0.75, 1.0)})
print(f'{steps_per_epoch} шагов за эпоху; NDCG на шагах {check_steps}', flush=True)

optimizer = torch.optim.AdamW((p for p in model.parameters() if p.requires_grad), lr=3e-5)
scaler = torch.amp.GradScaler('cuda')
model.train()
optimizer.zero_grad(set_to_none=True)

best_ndcg = -1.0
bad_checks = 0
steps = 0

for i, (tokens, label) in enumerate(zip(train_tokens, labels), start=1):
    ids = torch.tensor([tokens], dtype=torch.long, device=input_device)
    with torch.autocast('cuda', dtype=torch.float16):
        logits = model(input_ids=ids, logits_to_keep=1, use_cache=False).logits[0, -1]
        answer_logits = torch.stack((logits[no_id], logits[yes_id])).unsqueeze(0)
        target = torch.tensor([int(label)], device=answer_logits.device)
        loss = F.cross_entropy(answer_logits.float(), target)
    scaler.scale(loss / ACCUMULATION).backward()
    loss_value = loss.detach().item()
    del ids, logits, answer_logits, target, loss

    if i % ACCUMULATION and i < len(train_tokens):
        continue
    scaler.step(optimizer)
    scaler.update()
    optimizer.zero_grad(set_to_none=True)
    steps += 1

    if steps % 25 == 0:
        print(f'Шаг {steps}/{steps_per_epoch}, loss={loss_value:.4f}', flush=True)

    if steps in check_steps:
        score = validate()
        print(f'Шаг {steps}: NDCG@10 (96 запросов) = {score:.5f}', flush=True)
        if score > best_ndcg + MIN_DELTA:
            best_ndcg = score
            bad_checks = 0
            model.save_pretrained(OUT / 'qwen_best')
            print('Сохранён qwen_best', flush=True)
        else:
            bad_checks += 1
            if bad_checks >= PATIENCE:
                print('Early stopping: качество перестало улучшаться', flush=True)
                break

if best_ndcg < 0:
    model.save_pretrained(OUT / 'qwen_best')
print(f'Готово. Шагов: {steps}/{steps_per_epoch}, лучший NDCG@10: {best_ndcg:.5f}', flush=True)

In [ ]:
run('qwen_train.py')

## Yuanbao-Reranker-8B


In [ ]:
%%writefile "{work_dir}/yuanbao_train.py"
import gc
import math
import random
import re
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from peft import LoraConfig, TaskType, get_peft_model, prepare_model_for_kbit_training
from sklearn.model_selection import GroupShuffleSplit
from transformers import AutoTokenizer

from amml_common import YUANBAO, load_reranker, prepare

MODEL_NAME, REVISION = YUANBAO
TARGET_TRAIN_PAIRS = 18000
SEED = 42
MAX_LENGTH = 1024
ACCUMULATION = 8
LEARNING_RATE = 3e-5
OUTPUT = Path('yuanbao')
OUTPUT.mkdir(exist_ok=True)

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

In [ ]:
%%writefile -a "{work_dir}/yuanbao_train.py"
# Разделение на train и validation
train_all = prepare(pd.read_csv('data/train.csv'))
fit_index, validation_index = next(GroupShuffleSplit(
    n_splits=1, test_size=0.2, random_state=SEED
).split(train_all, groups=train_all.query_id))
fit_full = train_all.iloc[fit_index].reset_index(drop=True)
validation = train_all.iloc[validation_index].reset_index(drop=True)
rng_val = np.random.default_rng(SEED)
selection_ids = rng_val.choice(validation.query_id.unique(), size=96, replace=False)
val96 = validation[validation.query_id.isin(selection_ids)].reset_index(drop=True)

In [ ]:
%%writefile -a "{work_dir}/yuanbao_train.py"
def select_training_rows(data, target, seed):
    pos = data[data.relevance == 1]
    neg = data[data.relevance == 0].copy()
    required = target - len(pos)
    assert 0 < required <= len(neg)

    pattern = re.compile(r'[\wёЁ]{2,}', re.UNICODE)
    query_terms = {qid: set(pattern.findall(str(q).lower()))
                   for qid, q in data[['query_id', 'query']].drop_duplicates('query_id').itertuples(index=False, name=None)}
    neg['_overlap'] = [len(query_terms[qid] & set(pattern.findall(doc.lower())))
                       for qid, doc in zip(neg.query_id, neg.doc)]
    rng = np.random.default_rng(seed)
    neg['_rand'] = rng.random(len(neg))

    required_per_query = (neg.sort_values(['query_id','_overlap','_rand'],
                                          ascending=[True,False,True])
                          .groupby('query_id', sort=False).head(1))
    remaining = neg.drop(index=required_per_query.index)
    extra = required - len(required_per_query)
    hard_count = extra // 2
    hard = remaining.sort_values(['_overlap','_rand'],
                                 ascending=[False,True]).head(hard_count)
    pool = remaining.drop(index=hard.index)
    random_part = pool.sample(n=extra-hard_count, random_state=seed)

    chosen_neg = pd.concat([required_per_query, hard, random_part]).sort_index()
    selected = pd.concat([pos, chosen_neg[data.columns]]).sort_index().reset_index(drop=True)
    assert len(selected) == target
    return selected

In [ ]:
%%writefile -a "{work_dir}/yuanbao_train.py"
fit = select_training_rows(fit_full, TARGET_TRAIN_PAIRS, SEED)
print(f'Train: {len(fit)} пар')

In [ ]:
%%writefile -a "{work_dir}/yuanbao_train.py"
# Токенизация и промпт
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, revision=REVISION, trust_remote_code=False)
tokenizer.padding_side = 'left'
if tokenizer.pad_token_id is None:
    tokenizer.pad_token = tokenizer.eos_token

yes_tokens = tokenizer.encode('yes', add_special_tokens=False)
no_tokens  = tokenizer.encode('no', add_special_tokens=False)
assert len(yes_tokens) == len(no_tokens) == 1, (yes_tokens, no_tokens)
YES_ID, NO_ID = yes_tokens[0], no_tokens[0]
assert YES_ID != NO_ID

SYSTEM = (
    'Judge whether the Document meets the requirements based on the Query and the Instruct provided. '
    'Note that the answer can only be "yes" or "no".'
)
INSTRUCTION = 'Given a web search query, retrieve relevant passages that answer the query'
PREFIX = f'<|im_start|>system\n{SYSTEM}<|im_end|>\n<|im_start|>user\n'
SUFFIX = '<|im_end|>\n<|im_start|>assistant\n<think>\n\n</think>\n\n'
PREFIX_IDS = tokenizer.encode(PREFIX, add_special_tokens=False)
SUFFIX_IDS = tokenizer.encode(SUFFIX, add_special_tokens=False)
BODY_BUDGET = MAX_LENGTH - len(PREFIX_IDS) - len(SUFFIX_IDS)
assert BODY_BUDGET >= 256, 'Служебный промпт занял слишком много токенов'


def encode_pair(query, doc):
    content = f'<Instruct>: {INSTRUCTION}\n\n<Query>: {query}\n\n<Document>: {doc}'
    body = tokenizer.encode(
        content, add_special_tokens=False,
        truncation=True, max_length=BODY_BUDGET
    )
    return PREFIX_IDS + body + SUFFIX_IDS

train_tokens = [encode_pair(q, d) for q, d in zip(fit['query'], fit['doc'])]
val96_tokens = [encode_pair(q, d) for q, d in zip(val96['query'], val96['doc'])]
lengths = np.array([len(x) for x in train_tokens], dtype=int)

In [ ]:
%%writefile -a "{work_dir}/yuanbao_train.py"
# Загрузка модели и LoRA
model = load_reranker(MODEL_NAME, REVISION)
model = prepare_model_for_kbit_training(
    model, use_gradient_checkpointing=True,
    gradient_checkpointing_kwargs={'use_reentrant': False}
)
model = get_peft_model(model, LoraConfig(
    r=16, lora_alpha=32, lora_dropout=0.05, bias='none',
    target_modules=['q_proj','k_proj','v_proj','o_proj','gate_proj','up_proj','down_proj'],
    task_type=TaskType.CAUSAL_LM,
))

input_device = model.get_input_embeddings().weight.device

In [ ]:
%%writefile -a "{work_dir}/yuanbao_train.py"
# Веса классов и оптимизатор
n_pos = int((fit.relevance == 1).sum())
n_neg = int((fit.relevance == 0).sum())
CLASS_WEIGHTS = torch.tensor([
    len(fit) / (2 * n_neg), len(fit) / (2 * n_pos)
], dtype=torch.float32, device=input_device)

optimizer = torch.optim.AdamW(
    (p for p in model.parameters() if p.requires_grad),
    lr=LEARNING_RATE,
)
scaler = torch.amp.GradScaler('cuda')

INFER_BATCH = 2

In [ ]:
%%writefile -a "{work_dir}/yuanbao_train.py"
# NDCG и скоринг
def ndcg_at_10(frame, predictions):
    x = frame[['id', 'query_id', 'relevance']].copy()
    x['id'] = x['id'].astype(str)
    x['prediction'] = np.asarray(predictions, dtype=float)
    results = []
    for _, g in x.groupby('query_id', sort=False):
        ranked = g.sort_values(['prediction', 'id'], ascending=[False, True])
        y = ranked.relevance.to_numpy()[:10]
        ideal = np.sort(g.relevance.to_numpy())[::-1][:10]
        discounts = 1.0 / np.log2(np.arange(2, len(y) + 2))
        ideal_discounts = 1.0 / np.log2(np.arange(2, len(ideal) + 2))
        numerator = float(((2 ** y - 1) * discounts).sum())
        denominator = float(((2 ** ideal - 1) * ideal_discounts).sum())
        results.append(numerator / denominator if denominator else 0.0)
    return float(np.mean(results))


def _score_chunk(chunk):
    # Токены дополняются слева, поэтому ответ берём с последней позиции.
    batch = tokenizer.pad({'input_ids': chunk}, padding=True,
                          return_attention_mask=True, return_tensors='pt')
    batch = {k: v.to(input_device) for k, v in batch.items()}
    mask = batch['attention_mask']
    positions = (mask.long().cumsum(-1) - 1).masked_fill(mask == 0, 0)
    with torch.inference_mode(), torch.autocast('cuda', dtype=torch.float16):
        outputs = model(**batch, position_ids=positions,
                        logits_to_keep=1, use_cache=False)
        last_logits = outputs.logits[:, -1, :]
        margins = last_logits[:, YES_ID].float() - last_logits[:, NO_ID].float()
    values = margins.detach().cpu().numpy().astype(np.float64)
    return values.tolist()


def score_rows(token_lists):
    global INFER_BATCH
    previously_training = model.training
    model.eval()
    scores = []
    i = 0
    try:
        while i < len(token_lists):
            k = min(INFER_BATCH, len(token_lists) - i)
            try:
                values = _score_chunk(token_lists[i:i+k])
            except torch.cuda.OutOfMemoryError:
                if k == 1:
                    raise
                print('OOM при batch=2, переключаем инференс на batch=1', flush=True)
                INFER_BATCH = 1
                gc.collect()
                torch.cuda.empty_cache()
                continue
            scores.extend(values)
            i += k
    finally:
        if previously_training:
            model.train()
    result = np.asarray(scores, dtype=np.float64)
    assert len(result) == len(token_lists) and np.isfinite(result).all()
    return result

In [ ]:
%%writefile -a "{work_dir}/yuanbao_train.py"
train_order = np.random.default_rng(SEED).permutation(len(train_tokens))
train_labels = fit.relevance.to_numpy(dtype=np.int64)
longest = int(lengths.argmax())
train_order = np.concatenate(([longest], train_order[train_order != longest]))

TOTAL_STEPS = math.ceil(len(fit) / ACCUMULATION)
check_steps = sorted({max(1, round(TOTAL_STEPS * f)) for f in (0.25, 0.50, 0.75, 1.0)})
print('Проверки NDCG на шагах:', check_steps)

best_score = -1.0
completed_steps = 0

In [ ]:
%%writefile -a "{work_dir}/yuanbao_train.py"
# Оценка и сохранение checkpoint
def save_checkpoint_and_validate(step):
    global best_score
    metric_scores = score_rows(val96_tokens)
    metric = ndcg_at_10(val96, metric_scores)
    print(f'Шаг {step}/{TOTAL_STEPS}: NDCG@10 (96 запросов) = {metric:.6f}', flush=True)
    if metric > best_score:
        best_score = metric
        model.save_pretrained(OUTPUT / 'yuanbao_best', safe_serialization=True)
        print('  → Новый лучший адаптер', flush=True)

In [ ]:
%%writefile -a "{work_dir}/yuanbao_train.py"
# Основной цикл обучения
model.train()
optimizer.zero_grad(set_to_none=True)
loss_accumulator = 0.0

for block_start in range(0, len(train_order), ACCUMULATION):
    batch_indices = train_order[block_start:block_start + ACCUMULATION]
    n_in_block = len(batch_indices)
    for idx in batch_indices:
        idx = int(idx)
        ids = torch.tensor([train_tokens[idx]], dtype=torch.long, device=input_device)
        with torch.autocast('cuda', dtype=torch.float16):
            logits = model(input_ids=ids, logits_to_keep=1, use_cache=False).logits[0, -1]
            answer = torch.stack((logits[NO_ID], logits[YES_ID])).float()[None, :]
            label = int(train_labels[idx])
            target = torch.tensor([label], dtype=torch.long, device=answer.device)
            loss = F.cross_entropy(answer, target) * CLASS_WEIGHTS[label] / n_in_block
        scaler.scale(loss).backward()
        loss_accumulator += float(loss.detach())
        del ids, logits, answer, target, loss
    scaler.unscale_(optimizer)
    torch.nn.utils.clip_grad_norm_((p for p in model.parameters() if p.requires_grad), 1.0)
    scaler.step(optimizer)
    scaler.update()
    optimizer.zero_grad(set_to_none=True)
    completed_steps += 1

    if completed_steps % 25 == 0:
        print(f'Шаг {completed_steps}/{TOTAL_STEPS}, loss за 25 шагов={loss_accumulator:.4f}', flush=True)
        loss_accumulator = 0.0

    if completed_steps in check_steps:
        save_checkpoint_and_validate(completed_steps)

assert best_score >= 0, "Не удалось выбрать адаптер Yuanbao"

In [ ]:
run('yuanbao_train.py')

## Giga-Embeddings 3B


In [ ]:
%%writefile "{work_dir}/giga_train.py"
import json
import os
import re
import subprocess
import sys
from pathlib import Path
from tempfile import mkdtemp

import pandas as pd
import torch
from sklearn.model_selection import GroupShuffleSplit

from amml_common import GIGA, prepare

MODEL, REVISION = GIGA
WORK = Path('giga').resolve()
WORK.mkdir(exist_ok=True)
INSTRUCTION = "Given a web search query, retrieve relevant passages that answer the query"
SEED = 42


def overlap(query, text):
    words = {w for w in re.findall(r"\w+", query.casefold()) if len(w) > 2}
    return len(words & set(re.findall(r"\w+", text.casefold()))) / max(1, len(words))

In [ ]:
%%writefile -a "{work_dir}/giga_train.py"
# Формирование контрастивных примеров
def message(text, role):
    return [{"role": role, "content": text}]


def make_examples(frame):
    rows = []
    for _, group in frame.groupby("query_id", sort=True):
        positives = group[group.relevance == 1]
        negatives = group[group.relevance == 0].copy()
        if positives.empty or negatives.empty:
            continue
        query = group.iloc[0]["query"]
        negatives["lexical_score"] = negatives.doc.map(lambda d: overlap(query, d))
        negatives = negatives.sort_values(["lexical_score", "id"], ascending=[False, True])
        for positive in positives.itertuples():
            candidates = negatives[negatives.doc != positive.doc]
            if candidates.empty:
                continue
            same_article = candidates[candidates.article_id == positive.article_id]
            first = same_article.iloc[0] if len(same_article) else candidates.iloc[0]
            rest = candidates[candidates.id != first.id]
            second = rest.iloc[0] if len(rest) else first
            rows.append({
                "messages": message(f"Instruct: {INSTRUCTION}\nQuery: {query}", "user"),
                "positive_messages": [message(positive.doc, "assistant")],
                "negative_messages": [message(first.doc, "assistant"), message(second.doc, "assistant")],
            })
    return rows

In [ ]:
%%writefile -a "{work_dir}/giga_train.py"
# Разделение на train и validation
data = prepare(pd.read_csv('data/train.csv'))

splitter = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=SEED)
train_idx, _ = next(splitter.split(data, groups=data.query_id))
train = data.iloc[train_idx].copy()
examples = make_examples(train)
assert len(examples) == 6580, f"Неожиданное число примеров: {len(examples)}"
train_file = WORK / "giga_train.jsonl"
with train_file.open("w", encoding="utf-8") as f:
    for row in examples:
        f.write(json.dumps(row, ensure_ascii=False) + "\n")

print(f'Train: {len(examples)} примеров')

In [ ]:
%%writefile -a "{work_dir}/giga_train.py"
# Регистрация модели в ms-swift
register_file = WORK / "register_giga.py"
register_file.write_text('''from swift.model import Model, ModelGroup, ModelMeta, register_model
from swift.model.register import SentenceTransformersLoader
from swift.template import TemplateType

register_model(ModelMeta(
    "giga_embeddings",
    [ModelGroup([Model(
        "ai-sage/Giga-Embeddings-instruct-3B-0826",
        "ai-sage/Giga-Embeddings-instruct-3B-0826",
    )])],
    SentenceTransformersLoader,
    template=TemplateType.dummy,
    architectures=["Qwen3BidirectionalModel"],
))
''', encoding="utf-8")


In [ ]:
%%writefile -a "{work_dir}/giga_train.py"
# Подготовка запуска
gpu_count = min(torch.cuda.device_count(), 2)
assert gpu_count >= 1, "Для обучения Giga нужна CUDA GPU"

checkpoint_root = WORK / "giga_lora"
checkpoint_root.mkdir(exist_ok=True)
checkpoint_dir = Path(mkdtemp(prefix="run-", dir=checkpoint_root))
selected_file = WORK / "selected_checkpoint.txt"
selected_file.unlink(missing_ok=True)

train_env = os.environ.copy()
for key in ("RANK", "LOCAL_RANK", "WORLD_SIZE", "LOCAL_WORLD_SIZE", "MASTER_ADDR", "MASTER_PORT"):
    train_env.pop(key, None)
visible = os.environ.get("CUDA_VISIBLE_DEVICES", "0,1").split(",")
train_env.update({
    "CUDA_VISIBLE_DEVICES": ",".join(visible[:gpu_count]),
    "SWIFT_SINGLE_DEVICE_MODE": "0",
    "INFONCE_TEMPERATURE": "0.05",
    "INFONCE_USE_BATCH": "False",
    "INFONCE_HARD_NEGATIVES": "2",
})

In [ ]:
%%writefile -a "{work_dir}/giga_train.py"
# Команда обучения
command = [sys.executable]
if gpu_count == 2:
    command += [
        "-m", "torch.distributed.run", "--standalone", "--nnodes=1",
        "--nproc-per-node=2", "--local-addr=127.0.0.1",
    ]
command += [
    "-m", "swift.cli.sft",
    "--custom_register_path", str(register_file),
    "--model_type", "giga_embeddings", "--model", MODEL, "--use_hf", "true",
    "--model_revision", REVISION,
    "--task_type", "embedding", "--loss_type", "infonce",
    "--tuner_type", "lora", "--lora_rank", "16", "--lora_alpha", "32",
    "--target_modules", "all-linear",
    "--dataset", str(train_file), "--split_dataset_ratio", "0.0",
    "--num_train_epochs", "1", "--max_length", "512", "--truncation_strategy", "delete",
    "--per_device_train_batch_size", "1", "--gradient_accumulation_steps", str(16 // gpu_count),
    "--learning_rate", "5e-5", "--lr_scheduler_type", "cosine", "--warmup_ratio", "0.03",
    "--torch_dtype", "float16", "--attn_impl", "sdpa",
    "--gradient_checkpointing", "true",
    "--gradient_checkpointing_kwargs", '{"use_reentrant": false}',
    "--ddp_find_unused_parameters", "false",
    "--eval_strategy", "no", "--save_strategy", "steps" if gpu_count == 2 else "epoch", "--save_steps", "103",
    "--save_total_limit", "5", "--save_only_model", "false",
    "--logging_steps", "10", "--dataloader_num_workers", "0",
    "--seed", str(SEED), "--output_dir", str(checkpoint_dir), "--add_version", "false",
    "--report_to", "none",
]

subprocess.run(command, env=train_env, check=True)

In [ ]:
%%writefile -a "{work_dir}/giga_train.py"
# Адаптер текущего запуска
checkpoints = [
    path for path in checkpoint_dir.glob("checkpoint-*")
    if (path / "adapter_model.safetensors").is_file()
    and (path / "adapter_config.json").is_file()
]
if gpu_count == 2:
    checkpoint = checkpoint_dir / "checkpoint-384"
    if checkpoint not in checkpoints:
        found = ", ".join(sorted(path.name for path in checkpoints)) or "нет"
        raise RuntimeError(f"Ожидался checkpoint-384 Giga. Найдены: {found}.")
else:
    if not checkpoints:
        raise RuntimeError("Не найден адаптер Giga после обучения.")
    checkpoint = max(checkpoints, key=lambda path: int(path.name.rsplit("-", 1)[1]))

selected_file.write_text(str(checkpoint), encoding="utf-8")

In [ ]:
run('giga_train.py')

## Предсказания Qwen


In [ ]:
%%writefile "{work_dir}/qwen_infer.py"
# Параметры модели
from pathlib import Path
import torch
from transformers import AutoTokenizer
from peft import PeftModel

import numpy as np
import pandas as pd

from amml_common import QWEN, load_reranker, prepare

MODEL_NAME, REVISION = QWEN
MAX_LENGTH = 1024
ADAPTER_DIR = Path('qwen')/'qwen_best'
OUT = Path('predictions')

assert (ADAPTER_DIR / 'adapter_config.json').is_file()

test = prepare(pd.read_csv('data/test.csv', dtype={'id': str}))

In [ ]:
%%writefile -a "{work_dir}/qwen_infer.py"
# Промпт и токенизация
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, revision=REVISION)
no_id = tokenizer.convert_tokens_to_ids('no')
yes_id = tokenizer.convert_tokens_to_ids('yes')
assert isinstance(yes_id, int) and isinstance(no_id, int) and yes_id != no_id
assert tokenizer.decode([yes_id]).strip() == 'yes', 'Изменилась токенизация yes'
assert tokenizer.decode([no_id]).strip() == 'no', 'Изменилась токенизация no'

prefix = ('<|im_start|>system\nJudge whether the Document meets the requirements '
          'based on the Query and the Instruct provided. Note that the answer '
          'can only be "yes" or "no".<|im_end|>\n<|im_start|>user\n')
suffix = '<|im_end|>\n<|im_start|>assistant\n<think>\n\n</think>\n\n'
prefix_ids = tokenizer.encode(prefix, add_special_tokens=False)
suffix_ids = tokenizer.encode(suffix, add_special_tokens=False)
room = MAX_LENGTH - len(prefix_ids) - len(suffix_ids)
assert room > 256

def encode(query, doc):
    content = (f'<Instruct>: Given a web search query, retrieve relevant passages '
               f'that answer the query\n<Query>: {query}\n<Document>: {doc}')
    middle = tokenizer.encode(content, add_special_tokens=False, truncation=True, max_length=room)
    return prefix_ids + middle + suffix_ids

In [ ]:
%%writefile -a "{work_dir}/qwen_infer.py"
# Загрузка адаптера
base = load_reranker(MODEL_NAME, REVISION)
model = PeftModel.from_pretrained(base, str(ADAPTER_DIR), is_trainable=False)
model.eval()
input_device = model.get_input_embeddings().weight.device
assert str(input_device) == 'cuda:0'

In [ ]:
%%writefile -a "{work_dir}/qwen_infer.py"
# Расчёт скоров
@torch.inference_mode()
def predict_rows(frame):
    scores = np.empty(len(frame), dtype=np.float64)
    model.eval()
    for i, (query, doc) in enumerate(zip(frame['query'], frame['doc'])):
        ids = torch.tensor([encode(query, doc)], dtype=torch.long, device=input_device)
        with torch.autocast('cuda', dtype=torch.float16):
            logits = model(input_ids=ids, logits_to_keep=1, use_cache=False).logits[0, -1]
        scores[i] = (logits[yes_id] - logits[no_id]).float().item()
        del ids, logits
    assert np.isfinite(scores).all()
    return scores

In [ ]:
%%writefile -a "{work_dir}/qwen_infer.py"
# Сохранение тестовых предсказаний
preds=predict_rows(test)
result=test[['id','query_id']].copy()
result['prediction']=preds
assert result.id.is_unique and np.isfinite(result.prediction).all()
result.to_csv(OUT/'qwen_test.csv',index=False)

In [ ]:
run('qwen_infer.py')

## Предсказания Yuanbao


In [ ]:
%%writefile "{work_dir}/yuanbao_infer.py"
# Параметры модели
import gc
from pathlib import Path
import torch
from transformers import AutoTokenizer
from peft import PeftModel

import numpy as np
import pandas as pd

from amml_common import YUANBAO, load_reranker, prepare

MODEL_NAME, REVISION = YUANBAO
MAX_LENGTH = 1024
INFER_BATCH = 2
OUTPUT = Path('predictions')
ADAPTER_DIR = Path('yuanbao')/'yuanbao_best'
assert (ADAPTER_DIR / 'adapter_config.json').is_file()

test = prepare(pd.read_csv('data/test.csv', dtype={'id': str}))

In [ ]:
%%writefile -a "{work_dir}/yuanbao_infer.py"
# Промпт и токенизация
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, revision=REVISION, trust_remote_code=False)
tokenizer.padding_side = 'left'
if tokenizer.pad_token_id is None:
    tokenizer.pad_token = tokenizer.eos_token

yes_tokens = tokenizer.encode('yes', add_special_tokens=False)
no_tokens  = tokenizer.encode('no', add_special_tokens=False)
assert len(yes_tokens) == len(no_tokens) == 1, (yes_tokens, no_tokens)
YES_ID, NO_ID = yes_tokens[0], no_tokens[0]
assert YES_ID != NO_ID

SYSTEM = (
    'Judge whether the Document meets the requirements based on the Query and the Instruct provided. '
    'Note that the answer can only be "yes" or "no".'
)
INSTRUCTION = 'Given a web search query, retrieve relevant passages that answer the query'
PREFIX = f'<|im_start|>system\n{SYSTEM}<|im_end|>\n<|im_start|>user\n'
SUFFIX = '<|im_end|>\n<|im_start|>assistant\n<think>\n\n</think>\n\n'
PREFIX_IDS = tokenizer.encode(PREFIX, add_special_tokens=False)
SUFFIX_IDS = tokenizer.encode(SUFFIX, add_special_tokens=False)
BODY_BUDGET = MAX_LENGTH - len(PREFIX_IDS) - len(SUFFIX_IDS)
assert BODY_BUDGET >= 256, 'Служебный промпт занял слишком много токенов'


def encode_pair(query, doc):
    content = f'<Instruct>: {INSTRUCTION}\n\n<Query>: {query}\n\n<Document>: {doc}'
    body = tokenizer.encode(
        content, add_special_tokens=False,
        truncation=True, max_length=BODY_BUDGET
    )
    return PREFIX_IDS + body + SUFFIX_IDS

In [ ]:
%%writefile -a "{work_dir}/yuanbao_infer.py"
# Загрузка 4-битной модели
model = load_reranker(MODEL_NAME, REVISION)
model = PeftModel.from_pretrained(model, str(ADAPTER_DIR), is_trainable=False)
model.eval()

input_device = model.get_input_embeddings().weight.device
assert str(input_device).startswith('cuda:0'), f'Входы должны быть на GPU0: {input_device}'

In [ ]:
%%writefile -a "{work_dir}/yuanbao_infer.py"
# Расчёт скоров
def _score_chunk(chunk):
    # Токены дополняются слева, поэтому ответ берём с последней позиции
    batch = tokenizer.pad({'input_ids': chunk}, padding=True,
                          return_attention_mask=True, return_tensors='pt')
    batch = {k: v.to(input_device) for k, v in batch.items()}
    mask = batch['attention_mask']
    positions = (mask.long().cumsum(-1) - 1).masked_fill(mask == 0, 0)
    with torch.inference_mode(), torch.autocast('cuda', dtype=torch.float16):
        outputs = model(**batch, position_ids=positions,
                        logits_to_keep=1, use_cache=False)
        last_logits = outputs.logits[:, -1, :]
        margins = last_logits[:, YES_ID].float() - last_logits[:, NO_ID].float()
    values = margins.detach().cpu().numpy().astype(np.float64)
    return values.tolist()


def score_rows(token_lists):
    global INFER_BATCH
    previously_training = model.training
    model.eval()
    scores = []
    i = 0
    try:
        while i < len(token_lists):
            k = min(INFER_BATCH, len(token_lists) - i)
            try:
                values = _score_chunk(token_lists[i:i+k])
            except torch.cuda.OutOfMemoryError:
                if k == 1:
                    raise
                print('OOM при batch=2, переключаем инференс на batch=1', flush=True)
                INFER_BATCH = 1
                gc.collect()
                torch.cuda.empty_cache()
                continue
            scores.extend(values)
            i += k
    finally:
        if previously_training:
            model.train()
    result = np.asarray(scores, dtype=np.float64)
    assert len(result) == len(token_lists) and np.isfinite(result).all()
    return result

In [ ]:
%%writefile -a "{work_dir}/yuanbao_infer.py"
# Сохранение тестовых предсказаний
test_tokens=[encode_pair(q,d) for q,d in zip(test['query'],test['doc'])]
scores=score_rows(test_tokens)
result=test[['id','query_id']].copy();result['prediction']=scores
assert result.id.is_unique and np.isfinite(scores).all()
result.to_csv(OUTPUT/'yuanbao_test.csv',index=False)

In [ ]:
run('yuanbao_infer.py')

## Предсказания Giga


In [ ]:
%%writefile "{work_dir}/giga_infer.py"
# Параметры модели
import json
from pathlib import Path
import torch
from sentence_transformers import SentenceTransformer
from safetensors import safe_open
from peft import LoraConfig, inject_adapter_in_model

import numpy as np
import pandas as pd

from amml_common import GIGA, prepare

MODEL_NAME, REVISION = GIGA
INSTRUCTION = 'Given a web search query, retrieve relevant passages that answer the query'
MAX_LENGTH = 512
BATCH_SIZE = 2
OUT = Path('predictions')
giga_dir = Path('giga')
CHECKPOINT = Path((giga_dir / 'selected_checkpoint.txt').read_text(encoding='utf-8').strip())
assert (CHECKPOINT / 'adapter_model.safetensors').is_file()
torch.set_grad_enabled(False)

test = prepare(pd.read_csv('data/test.csv', dtype={'id': str}))

In [ ]:
%%writefile -a "{work_dir}/giga_infer.py"
# Базовая модель и скоринг
model = SentenceTransformer(
    MODEL_NAME, revision=REVISION, trust_remote_code=True, device='cuda:0',
    model_kwargs={'torch_dtype': torch.float16},
)
model.max_seq_length = MAX_LENGTH
model.eval()


def score_pairs(frame, batch_size=BATCH_SIZE):
    qrows = frame[['query_id', 'query']].drop_duplicates('query_id')
    query_inputs = [f'Instruct: {INSTRUCTION}\nQuery: {q}' for q in qrows['query']]
    docs = frame['doc'].drop_duplicates().tolist()
    with torch.inference_mode():
        qvec = model.encode(query_inputs, batch_size=batch_size, normalize_embeddings=True, show_progress_bar=True)
        dvec = model.encode(docs, batch_size=batch_size, normalize_embeddings=True, show_progress_bar=True)
    qi = pd.Series(np.arange(len(qrows)), index=qrows.query_id).to_dict()
    di = {s: i for i, s in enumerate(docs)}
    qids = np.array([qi[q] for q in frame.query_id], dtype=int)
    dids = np.array([di[d] for d in frame.doc], dtype=int)
    scores = (qvec[qids].astype(np.float32) * dvec[dids].astype(np.float32)).sum(axis=1)
    return scores

In [ ]:
%%writefile -a "{work_dir}/giga_infer.py"
# Подготовка LoRA
adapter_path = CHECKPOINT / 'adapter_model.safetensors'
config_path = CHECKPOINT / 'adapter_config.json'
adapter_cfg = json.loads(config_path.read_text(encoding='utf-8'))
assert int(adapter_cfg.get('r', -1)) == 16
assert int(adapter_cfg.get('lora_alpha', -1)) == 32
assert not adapter_cfg.get('rank_pattern'), 'Этот скрипт не поддерживает rank_pattern'
assert not adapter_cfg.get('alpha_pattern'), 'Этот скрипт не поддерживает alpha_pattern'
assert adapter_cfg.get('bias', 'none') == 'none', 'Обнаружен LoRA с дополнительными bias'

TARGETS = ['q_proj', 'k_proj', 'v_proj', 'o_proj', 'gate_proj', 'up_proj', 'down_proj']
inner_model = model[0].auto_model
lora_config = LoraConfig(
    r=16, lora_alpha=32, target_modules=TARGETS,
    lora_dropout=0.0, bias='none', task_type=None,
)

In [ ]:
%%writefile -a "{work_dir}/giga_infer.py"
# Загрузка тензоров LoRA
inject_adapter_in_model(lora_config, inner_model, adapter_name='default')

parameters = dict(inner_model.named_parameters())
expected = {key for key in parameters if '.lora_A.default.weight' in key or '.lora_B.default.weight' in key}
assert len(expected) == 504, f'Ожидали 504 LoRA-тензора, создано {len(expected)}'

PREFIX = 'base_model.model.0.auto_model.'
loaded = set()
with safe_open(str(adapter_path), framework='pt', device='cpu') as sf:
    original_keys = list(sf.keys())
    assert len(original_keys) == 504, f'В файле {len(original_keys)} тензоров вместо 504'
    with torch.no_grad():
        for original_name in original_keys:
            if not original_name.startswith(PREFIX):
                raise ValueError('Неожиданный префикс: ' + original_name)
            target = original_name[len(PREFIX):]
            target = target.replace('.lora_A.weight', '.lora_A.default.weight')
            target = target.replace('.lora_B.weight', '.lora_B.default.weight')
            if target not in expected:
                raise KeyError('LoRA-модуль не найден в модели: ' + target)
            tensor = sf.get_tensor(original_name)
            param = parameters[target]
            if tuple(tensor.shape) != tuple(param.shape):
                raise ValueError(f'Несовпадение размеров {target}: {tensor.shape} vs {param.shape}')
            param.copy_(tensor.to(device=param.device, dtype=param.dtype))
            loaded.add(target)

assert loaded == expected, f"Не загружены {len(expected - loaded)} LoRA-тензоров"

In [ ]:
%%writefile -a "{work_dir}/giga_infer.py"
# Сохранение тестовых предсказаний
model.eval()
test_scores=score_pairs(test)
result=test[['id','query_id']].copy()
result['prediction']=test_scores.astype(float)
assert len(result)==13507 and result.id.is_unique and np.isfinite(result.prediction).all()
result.to_csv(OUT/'giga_test.csv',index=False)

In [ ]:
run('giga_infer.py')

## Ансамбль


In [ ]:
# Нормализация рангов
import numpy as np

def model_ranks(frame, score_col):
    ordered=frame[['id','query_id',score_col]].sort_values(
        ['query_id',score_col,'id'],ascending=[True,False,True],kind='mergesort').copy()
    ordered['r']=ordered.groupby('query_id',sort=False).cumcount()+1
    ordered['n']=ordered.groupby('query_id',sort=False)['id'].transform('size')
    ordered['normalized'] = np.where(ordered['n']>1,
                                    (ordered['n']-ordered['r'])/(ordered['n']-1).clip(lower=1),1.0)
    return ordered.set_index('id')['normalized'].reindex(frame['id']).to_numpy(dtype=float)

In [ ]:
# Объединение предсказаний
source=test_raw[['id','query_id']].copy()
source['id']=source.id.astype(str)
for key,filename in [('qwen','qwen_test.csv'),('yuanbao','yuanbao_test.csv'),('giga','giga_test.csv')]:
    pred=pd.read_csv(pred_dir/filename,dtype={'id':str})
    source=source.merge(pred[['id','prediction']].rename(columns={'prediction':key}),on='id',validate='one_to_one',sort=False)
assert len(source) == len(test_raw)

In [ ]:
# Веса моделей
yr=model_ranks(source,'yuanbao')
qr=model_ranks(source,'qwen')
gr=model_ranks(source,'giga')
score=0.50*yr+0.30*qr+0.20*gr

In [ ]:
# Итоговый файл
submission=pd.DataFrame({'id':source.id.astype(str),'prediction':score})
final_path=work_dir/'submission.csv'
assert np.isfinite(submission.prediction).all()
submission.to_csv(final_path, index=False, float_format='%.12f')
print('Сохранено:', final_path)